# Midterm Machine Learning — Song Release Year Prediction

| Info | Detail |
|------|--------|
| **Nama** | Najwa Bilqis Al Khalidah |
| **NIM** | 101032300186 |
| **Kelas** | TK-46-GAB |

---

## Tujuan
Membangun pipeline end-to-end machine learning regresi untuk memprediksi tahun rilis sebuah lagu berdasarkan 90 fitur audio (timbre dan karakteristik sinyal musik).

## Alur Pipeline
```
Load Data -> EDA -> Feature Engineering -> Preprocessing
    -> Model Training (Ridge, RandomForest, XGBoost+Optuna)
    -> Evaluasi (MSE, RMSE, MAE, R2) -> LIME -> MLflow -> Simpan Model
```

## Cell 1 — Install & Import Library

Install library tambahan yang tidak tersedia secara default di Google Colab:
- **optuna**: framework hyperparameter tuning berbasis Bayesian optimization
- **mlflow**: experiment tracking (log parameter, metrik, dan model artifact)
- **lime**: model explainability — menjelaskan prediksi secara lokal per sampel

XGBoost dikonfigurasi dengan `tree_method='hist'` tanpa `device='cuda'` agar aman di semua runtime Colab (CPU maupun GPU). Optuna dibatasi 15 trial untuk efisiensi.

In [ ]:
!pip install optuna mlflow lime -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import VarianceThreshold
import xgboost as xgb
import optuna
import mlflow
import mlflow.sklearn
import lime
import lime.lime_tabular
import joblib
import os

try:
    import subprocess
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode()
    USE_GPU = True
    print('GPU tersedia — XGBoost akan menggunakan device=cuda')
    print(gpu_info.split('\n')[0])
except Exception:
    USE_GPU = False
    print('GPU tidak terdeteksi — menggunakan CPU (tree_method=hist)')

XGB_DEVICE = 'cuda' if USE_GPU else 'cpu'
print(f'Semua library berhasil diimport. XGB device: {XGB_DEVICE}')

## Cell 2 — Load Dataset

Dataset `midterm-regresi-dataset.csv` berisi:
- **Kolom pertama**: target label — tahun rilis lagu (contoh: 2001)
- **Kolom 2–91**: 90 fitur numerik audio hasil komputasi dari sinyal musik (timbre, dll.)

Dataset ini tidak memiliki header, sehingga kolom di-rename secara manual: kolom pertama menjadi `year`, sisanya menjadi `feature_1` sampai `feature_90`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE_PATH = '/content/drive/MyDrive/ML Dataset/'

print('Loading dataset...')
df = pd.read_csv(BASE_PATH + 'midterm-regresi-dataset.csv', header=None)

df.columns = ['year'] + [f'feature_{i}' for i in range(1, 91)]

print(f'Shape: {df.shape}')
print(f'\nContoh 3 baris pertama:')
display(df.head(3))
print(f'\nTarget (year) stats:')
print(df['year'].describe())

## Cell 3 — Exploratory Data Analysis (EDA)

EDA bertujuan memahami karakteristik data sebelum pemodelan:
1. **Distribusi target (`year`)** — apakah data terdistribusi merata atau skewed?
2. **Missing values** — apakah ada data yang hilang?
3. **Outlier** — boxplot untuk melihat sebaran nilai ekstrem
4. **Korelasi fitur dengan target** — fitur mana yang paling berkorelasi dengan tahun rilis?

Informasi ini memandu keputusan feature engineering di langkah berikutnya.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

df['year'].plot(kind='hist', bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Tahun Rilis Lagu')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Frekuensi')

df['year'].plot(kind='box', ax=axes[1], color='steelblue')
axes[1].set_title('Boxplot Tahun Rilis')

corr = df.corr()['year'].drop('year').abs().nlargest(15)
corr.plot(kind='barh', ax=axes[2], color='steelblue')
axes[2].set_title('Top 15 Fitur (Korelasi |r| vs Year)')
axes[2].set_xlabel('|Pearson r|')

plt.tight_layout()
plt.show()

print(f'Tahun paling tua : {df["year"].min()}')
print(f'Tahun paling baru: {df["year"].max()}')
print(f'Missing values   : {df.isnull().sum().sum()}')
print(f'\nDistribusi tahun terkonsentrasi di era 1990-2010 (skewed left).')
print(f'Model kemungkinan lebih akurat untuk lagu era modern.')

## Cell 4 — Feature Engineering

Feature engineering adalah proses membuat atau memilih fitur yang lebih informatif untuk meningkatkan performa model. Tiga langkah yang dilakukan:

1. **Variance Thresholding** — hapus fitur dengan varians sangat rendah (hampir konstan, tidak informatif). Threshold = 0.01.
2. **Correlation-based selection** — pertahankan fitur dengan |korelasi| >= 0.05 terhadap target `year`. Fitur yang tidak berkorelasi dengan target cenderung menambah noise.
3. **Interaction features** — buat fitur interaksi dari 5 fitur teratas (perkalian pasangan). Fitur interaksi menangkap hubungan non-linear antar fitur yang mungkin relevan terhadap tahun.

Dari 90 fitur, tidak semuanya relevan. Membuang fitur tidak berguna mengurangi dimensi, mempercepat training, dan meningkatkan generalisasi.

In [ ]:
X_raw = df.drop(columns=['year'])
y = df['year']

print(f'Jumlah fitur awal: {X_raw.shape[1]}')

vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X_raw)
selected_by_var = X_raw.columns[vt.get_support()].tolist()
print(f'Setelah variance threshold (>0.01): {len(selected_by_var)} fitur')

X_temp = pd.DataFrame(X_vt, columns=selected_by_var)
corr_with_target = X_temp.corrwith(y).abs()
selected_by_corr = corr_with_target[corr_with_target >= 0.05].index.tolist()
print(f'Setelah correlation filter (|r|>=0.05): {len(selected_by_corr)} fitur')

X = X_temp[selected_by_corr].copy()

top5 = corr_with_target[selected_by_corr].nlargest(5).index.tolist()
print(f'\nTop 5 fitur untuk interaction terms: {top5}')

interaction_names = []
for i in range(len(top5)):
    for j in range(i+1, len(top5)):
        col_name = f'inter_{top5[i]}_{top5[j]}'
        X[col_name] = X_temp[top5[i]] * X_temp[top5[j]]
        interaction_names.append(col_name)

print(f'Interaction features ditambahkan: {len(interaction_names)}')
print(f'\nTotal fitur setelah feature engineering: {X.shape[1]}')

feature_names = X.columns.tolist()

fig, ax = plt.subplots(figsize=(10, 3))
corr_with_target[selected_by_corr].sort_values(ascending=False).plot(
    kind='bar', ax=ax, color='steelblue', edgecolor='white'
)
ax.set_title('Korelasi Fitur Terpilih dengan Target (year)')
ax.set_xlabel('Fitur')
ax.set_ylabel('|Pearson r|')
ax.tick_params(axis='x', rotation=90, labelsize=7)
plt.tight_layout()
plt.show()

## Cell 5 — Preprocessing: Split & Scaling

Setelah feature engineering, dilakukan:
1. **Train-Validation Split** (80:20, random_state=42) — 80% untuk training, 20% untuk evaluasi generalisasi
2. **StandardScaler** — normalisasi fitur ke mean=0, std=1. Ini wajib untuk Ridge Regression agar regularisasi L2 adil di semua fitur. Untuk XGBoost dan Random Forest, scaling tidak wajib karena tree-based, namun tetap diterapkan untuk konsistensi pipeline.

Scaler di-fit hanya pada training set untuk menghindari data leakage ke validation set.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}   | y_val  : {y_val.shape}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

assert not np.isnan(X_train_scaled).any(), 'NaN ditemukan di training data!'
assert not np.isnan(X_val_scaled).any(), 'NaN ditemukan di validation data!'

print(f'\nPreprocessing selesai.')
print(f'Train mean (post-scale) = {X_train_scaled.mean():.4f}')
print(f'Train std  (post-scale) = {X_train_scaled.std():.4f}')

## Cell 6 — Fungsi Evaluasi & Setup MLflow

Fungsi `evaluate()` menghitung dan menampilkan 4 metrik regresi standar:
- **MSE** (Mean Squared Error) — rata-rata kuadrat error; sensitif terhadap outlier
- **RMSE** (Root MSE) — interpretasi lebih intuitif karena dalam unit yang sama dengan target (tahun)
- **MAE** (Mean Absolute Error) — rata-rata error absolut; lebih robust terhadap outlier
- **R2** (Coefficient of Determination) — proporsi variansi target yang dijelaskan model (0-1, makin tinggi makin baik)

MLflow experiment bernama `SongYear_Regression` diinisialisasi untuk tracking semua run.

In [ ]:
def evaluate(model_name, y_true, y_pred):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f}  (rata-rata meleset ±{rmse:.1f} tahun)")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R2   : {r2:.4f}")
    print(f"{'='*45}")

    return {'Model': model_name, 'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

results = []

mlflow.set_experiment('SongYear_Regression')
print('MLflow experiment SongYear_Regression siap.')

## Cell 7 — Model 1: Ridge Regression (Baseline)

**Ridge Regression** adalah model linear dengan regularisasi L2. Digunakan sebagai baseline karena:
- Sederhana dan cepat dilatih
- Memberikan benchmark minimal yang harus dilampaui model kompleks
- Regularisasi L2 (alpha=1.0) mencegah overfitting dengan menekan koefisien besar

Input menggunakan data yang sudah di-scale (`X_train_scaled`) karena Ridge sensitif terhadap skala fitur.

In [ ]:
with mlflow.start_run(run_name='Ridge_Baseline'):
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train)

    y_pred_ridge = ridge.predict(X_val_scaled)
    metrics_ridge = evaluate('Ridge Regression (Baseline)', y_val, y_pred_ridge)
    results.append(metrics_ridge)

    mlflow.log_param('alpha', 1.0)
    mlflow.log_param('model_type', 'Ridge')
    mlflow.log_metrics({k: v for k, v in metrics_ridge.items() if k != 'Model'})
    mlflow.sklearn.log_model(ridge, 'ridge_model')

print('Ridge Regression selesai dan ter-log ke MLflow.')
print('Ridge adalah model linear dengan performa terbatas untuk hubungan non-linear.')

## Cell 8 — Model 2: Random Forest Regressor

**Random Forest** adalah ensemble dari banyak decision tree yang dilatih secara paralel pada subset data dan fitur yang berbeda-beda (bagging). Keunggulannya:
- Menangkap hubungan non-linear yang tidak bisa dilakukan Ridge
- Robust terhadap outlier
- Memberikan feature importance secara built-in

Parameter yang digunakan moderat (`n_estimators=100`, `n_jobs=-1`) agar tidak terlalu berat di Colab. Random Forest tidak membutuhkan scaling, sehingga menggunakan `X_train` langsung.

Model ini berfungsi sebagai perbandingan antara Ridge (linear) dan XGBoost (gradient boosting).

In [ ]:
with mlflow.start_run(run_name='RandomForest_Baseline'):
    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    )
    rf.fit(X_train, y_train)

    y_pred_rf = rf.predict(X_val)
    metrics_rf = evaluate('Random Forest', y_val, y_pred_rf)
    results.append(metrics_rf)

    mlflow.log_params({
        'n_estimators': 100, 'max_depth': 12,
        'min_samples_leaf': 5, 'model_type': 'RandomForest'
    })
    mlflow.log_metrics({k: v for k, v in metrics_rf.items() if k != 'Model'})
    mlflow.sklearn.log_model(rf, 'rf_model')

feat_imp_rf = pd.Series(rf.feature_importances_, index=feature_names)
feat_imp_rf.nlargest(15).plot(kind='barh', figsize=(8, 5), color='coral')
plt.title('Top 15 Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Random Forest selesai.')

## Cell 9 — Model 3: XGBoost + Optuna Hyperparameter Tuning

**XGBoost** (Extreme Gradient Boosting) adalah algoritma gradient boosting yang membangun tree secara sekuensial, di mana setiap tree baru memperbaiki error dari tree sebelumnya.

**Optuna** digunakan untuk mencari kombinasi hyperparameter terbaik secara otomatis menggunakan algoritma Tree-structured Parzen Estimator (TPE) — lebih efisien dari grid search:
- `n_estimators`: jumlah tree (100-400)
- `max_depth`: kedalaman maksimal tree (3-7)
- `learning_rate`: kecepatan belajar (0.01-0.3, log scale)
- `subsample`: proporsi sampel per tree (0.6-1.0)
- `colsample_bytree`: proporsi fitur per tree (0.6-1.0)
- `min_child_weight`: minimum jumlah sampel di leaf (1-10)

Dibatasi 15 trial dengan early stopping 30 round dan `tree_method='hist'` untuk efisiensi di Colab.

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 400),
        'max_depth'       : trial.suggest_int('max_depth', 3, 7),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'tree_method'     : 'hist',
        'device'          : XGB_DEVICE,
        'random_state'    : 42,
        'early_stopping_rounds': 30,
    }

    with mlflow.start_run(run_name=f'XGB_trial_{trial.number}', nested=True):
        model = xgb.XGBRegressor(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        mlflow.log_params(params)
        mlflow.log_metric('val_rmse', rmse)

    return rmse

print('Mulai Optuna tuning XGBoost (15 trials)...')
optuna.logging.set_verbosity(optuna.logging.WARNING)

with mlflow.start_run(run_name='XGBoost_Optuna'):
    study_xgb = optuna.create_study(direction='minimize')
    study_xgb.optimize(objective_xgb, n_trials=15, show_progress_bar=True)

    print(f'\nBest RMSE : {study_xgb.best_value:.4f}')
    print(f'Best params : {study_xgb.best_params}')
    mlflow.log_metric('best_val_rmse', study_xgb.best_value)

print('\nOptuna tuning selesai.')

## Cell 10 — Training Final XGBoost & Feature Importance

Setelah mendapatkan hyperparameter terbaik dari Optuna, model final dilatih ulang dengan parameter tersebut pada seluruh training data.

**Feature Importance XGBoost** menunjukkan fitur mana yang paling sering digunakan untuk membuat split di seluruh tree. Fitur dengan importance tinggi adalah yang paling informatif untuk memprediksi tahun rilis.

In [ ]:
best_params = study_xgb.best_params.copy()
best_params.update({
    'tree_method' : 'hist',
    'device'      : XGB_DEVICE,
    'random_state': 42
})
best_params.pop('early_stopping_rounds', None)

print('Training final XGBoost dengan best params...')
final_xgb = xgb.XGBRegressor(**best_params)
final_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred_xgb = final_xgb.predict(X_val)
metrics_xgb = evaluate('XGBoost (Tuned)', y_val, y_pred_xgb)
results.append(metrics_xgb)

with mlflow.start_run(run_name='XGBoost_Final'):
    mlflow.log_params(best_params)
    mlflow.log_metrics({k: v for k, v in metrics_xgb.items() if k != 'Model'})
    mlflow.sklearn.log_model(final_xgb, 'xgb_final_model')

feat_imp_xgb = pd.Series(final_xgb.feature_importances_, index=feature_names)
top20 = feat_imp_xgb.nlargest(20)

fig, ax = plt.subplots(figsize=(8, 7))
top20.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Feature Importance — XGBoost (Tuned)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print(f'\nFitur terpenting: {top20.index[0]} (score: {top20.iloc[0]:.4f})')
print('\nFinal XGBoost selesai.')

## Cell 11 — Interpretasi Model dengan LIME

**LIME (Local Interpretable Model-agnostic Explanations)** menjelaskan prediksi sebuah model black-box untuk satu sampel tertentu dengan cara:
1. Membuat perturbasi lokal di sekitar sampel
2. Melatih model linear sederhana (interpretable) pada perturbasi tersebut
3. Koefisien model linear itu menjadi penjelasan lokal — seberapa besar setiap fitur mendorong atau menurunkan prediksi

Cara membaca output LIME:
- Batang positif (hijau) menandakan fitur mendorong prediksi ke tahun yang lebih tinggi
- Batang negatif (merah) menandakan fitur mendorong prediksi ke tahun yang lebih rendah
- Panjang batang menunjukkan besarnya pengaruh

Karena explainer dibangun dengan data scaled sementara `final_xgb` menerima data unscaled, digunakan wrapper `predict_fn` yang melakukan inverse transform sebelum memanggil model.

In [ ]:
print('Setup LIME explainer...')

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data = X_train_scaled,
    feature_names = feature_names,
    mode          = 'regression',
    random_state  = 42
)

def predict_fn(x_scaled):
    x_unscaled = scaler.inverse_transform(x_scaled)
    x_df = pd.DataFrame(x_unscaled, columns=feature_names)
    return final_xgb.predict(x_df)

for sample_idx in [0, 10, 50]:
    sample     = X_val_scaled[sample_idx]
    actual     = y_val.iloc[sample_idx]
    predicted  = predict_fn(sample.reshape(1, -1))[0]

    print(f'\nSampel #{sample_idx}')
    print(f'Actual year   : {actual}')
    print(f'Predicted year: {predicted:.2f}')
    print(f'Error         : {abs(actual - predicted):.2f} tahun')

    exp = explainer.explain_instance(
        data_row   = sample,
        predict_fn = predict_fn,
        num_features = 10
    )

    fig = exp.as_pyplot_figure()
    plt.title(f'LIME Explanation — Sampel #{sample_idx} | Actual: {actual} | Pred: {predicted:.1f}')
    plt.tight_layout()
    plt.show()

print('\nLIME selesai.')

## Cell 12 — Perbandingan Semua Model

Perbandingan performa ketiga model secara visual dan tabular.

Interpretasi metrik:
- **RMSE & MAE**: semakin kecil semakin baik — menunjukkan rata-rata kesalahan prediksi dalam satuan tahun
- **R2**: semakin besar semakin baik — proporsi variansi data yang berhasil dijelaskan model

Prediksi tahun rilis dari fitur audio adalah tugas yang secara inheren sulit karena hubungan fitur audio dengan tahun tidak selalu linear atau deterministik — gaya musik berevolusi perlahan, dan fitur audio yang sama bisa muncul di berbagai dekade.

In [ ]:
results_df = pd.DataFrame(results)
print('\n=== PERBANDINGAN SEMUA MODEL ===')
print(results_df.to_string(index=False))

best_model = results_df.loc[results_df['RMSE'].idxmin(), 'Model']
best_r2    = results_df['R2'].max()
print(f'\nModel terbaik: {best_model}')
print(f'R2 terbaik   : {best_r2:.4f} — model menjelaskan {best_r2*100:.1f}% variansi data')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_config = [
    ('RMSE', 'salmon',        'Semakin kecil = semakin baik'),
    ('MAE',  'steelblue',     'Semakin kecil = semakin baik'),
    ('R2',   'mediumseagreen','Semakin besar = semakin baik'),
]

for ax, (metric, color, note) in zip(axes, metrics_config):
    bars = ax.bar(results_df['Model'], results_df[metric], color=color, edgecolor='white')
    ax.set_title(f'Perbandingan {metric}\n({note})', fontsize=10)
    ax.set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Perbandingan Performa Model — Song Year Prediction', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_val, y_pred_xgb, alpha=0.3, s=10, color='steelblue', label='Prediksi XGBoost')
lims = [y_val.min(), y_val.max()]
ax.plot(lims, lims, 'r--', lw=1.5, label='Prediksi sempurna')
ax.set_xlabel('Actual Year')
ax.set_ylabel('Predicted Year')
ax.set_title('Actual vs Predicted — XGBoost (Tuned)')
ax.legend()
plt.tight_layout()
plt.show()

print('\nGrafik perbandingan dan scatter plot selesai.')

## Cell 13 — Simpan Model ke Google Drive

Semua artifact disimpan ke Google Drive agar dapat digunakan kembali tanpa training ulang:
- `ridge_model.pkl` — model Ridge Regression
- `rf_model.pkl` — model Random Forest
- `xgb_final.pkl` — model XGBoost terbaik
- `scaler.pkl` — StandardScaler yang di-fit pada training data
- `feature_names.pkl` — daftar nama fitur yang digunakan (penting untuk reproduksi)

Cara load ulang: `model = joblib.load('path/xgb_final.pkl')` kemudian `scaler = joblib.load('path/scaler.pkl')`

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/ML Dataset/models/'
os.makedirs(SAVE_PATH, exist_ok=True)

artifacts = {
    'ridge_model.pkl'    : ridge,
    'rf_model.pkl'       : rf,
    'xgb_final.pkl'      : final_xgb,
    'scaler.pkl'         : scaler,
    'feature_names.pkl'  : feature_names,
}

for filename, obj in artifacts.items():
    path = SAVE_PATH + filename
    joblib.dump(obj, path)
    print(f'Disimpan: {filename}')

print(f'\nSemua model dan artifact berhasil disimpan ke:')
print(f'   {SAVE_PATH}')

print('\n=== RINGKASAN AKHIR ===')
print(results_df.to_string(index=False))
print(f'\nModel terbaik: {best_model} (RMSE={results_df["RMSE"].min():.4f}, R2={best_r2:.4f})')